# Alignment of ORF sequences to cDNA sequences using minimap2

## First create an index for minimap2


In [ ]:
%%bash

minimap2 -x map-ont -d "/media/niek/4TB_SSD2/analyses/GPS_ONT/gencode_transcriptome.mmi" gencode.v49.pc_transcripts.fa.gz

[M::mm_idx_gen::14.165*1.57] collected minimizers
[M::mm_idx_gen::17.142*1.81] sorted minimizers
[M::main::18.972*1.74] loaded/built the index for 245535 target sequence(s)
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 245535
[M::mm_idx_stat::19.201*1.73] distinct minimizers: 14406706 (26.60% are singletons); average occurrences: 8.272; average spacing: 5.382; total length: 641403928
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -x map-ont -d /media/niek/4TB_SSD2/analyses/GPS_ONT/gencode_transcriptome.mmi gencode.v49.pc_transcripts.fa.gz
[M::main] Real time: 19.350 sec; CPU: 33.319 sec; Peak RSS: 3.668 GB


## Align the ORF sequences to the cDNA sequences using minimap2

Alignn with Minimap2, convert to BAM format and sort by RNAME (i.e. mapped reference sequence, cDNA):

In [ ]:
%%bash

REF="/media/niek/4TB_SSD2/analyses/GPS_ONT/gencode_transcriptome.mmi"
READS="/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_barcode_description.fastq.gz"
OUTPUT_BAM="/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_aligned.bam"

# Only keep primary alignments (i.e. best hit) and sort the output BAM file.
# The -y flag will move the BC:Z: tag to the read name.
# The --eqx flag will use extended CIGAR strings with = and X for matches and mismatches.
minimap2 -ax map-ont --eqx -y -t 8 --secondary=no $REF $READS | \
    samtools view -u - | \
    samtools sort -@ 4 -o $OUTPUT_BAM

[M::main::2.204*0.95] loaded/built the index for 245535 target sequence(s)
[M::mm_mapopt_update::2.616*0.96] mid_occ = 265
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 245535
[M::mm_idx_stat::2.910*0.96] distinct minimizers: 14406706 (26.60% are singletons); average occurrences: 8.272; average spacing: 5.382; total length: 641403928
[M::worker_pipeline::9.043*4.95] mapped 10000 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -y -ax map-ont -t 8 --secondary=no /media/niek/4TB_SSD2/analyses/GPS_ONT/gencode_transcriptome.mmi /media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_barcode_header.fastq.gz
[M::main] Real time: 9.118 sec; CPU: 44.803 sec; Peak RSS: 1.787 GB
[bam_sort_core] merging from 0 files and 4 in-memory blocks...


In [6]:
%%bash

OUTPUT_BAM="/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_aligned.bam"

samtools view $OUTPUT_BAM | head 

09ae93c5-266d-4aae-ad16-e62dd72392ec	0	ENST00000905711.1|ENSG00000160087.22|OTTHUMG00000001911.9|-|UBE2J2-227|UBE2J2|2677|UTR5:1-606|CDS:607-1386|UTR3:1387-2677|	607	0	188M1I590M2S	*	0	0	ATGAGCAGCACCAGCAGTAAGAGGGCTCCGACCACGGCAACCCAGAGGCTGAAGCAGGACTACCTTCGCATTAAGAAAGACCCGGTGCCTTACATCTGTGCCGAGCCCCTCCCTTCGAATATTCTCGAGTGGCACTATGTCGTCCGAGGCCCAGAGATGACCCCTTATGAAGGTGGCTATTATCATGGAAAAACTAATTTTTCCCAGAGAATTTCCTTTCAAACCTCCCAGTATCTATATGATCACTCCCAACGGGAGGTTTAAGTGCAACACCAGGCTGTGTCTTTCTATCACGGATTTCCACCCGGACACGTGGAACCCGGCCTGGTCTGTCTCCACCATCCTGACTGGGCTCCTGAGCTTCATGGTGGAGAAGGGCCCCACCCTGGGCAGTATAGAGACGTCGGACTTCACGAAAAGACAACTGGCAGTGCAGAGTTTAGCATTTAATTTGAAAGATAAAGTCTTTTGTGAATTATTTCCTGAAGTCGTGGAGGAGATTAAACAAAAACAGAAAGCACAAGACGAACTCAGTAGCAGACCCCAGACTCTCCCCTTGCCAGACGTGGTTCCAGACGGGGAGACGCACCTCGTCCAGAACGGGATTCAGCTGCTCAACGGGCATGCGCCGGGGGCCGTCCCAAACCTCGCAGGGCTCCAGCAGGCCAACCGGCACCACGGACTCCTGGGTGGCGCCCTGGCGAACTTGTTTGTGATAGTTGGGTTTGCAGCCTTTGGGTACACGGTCAAGTACGTGCTGAGGAGCATCGCGCAGGAGTAG	EEGFGFEEDEEGHECAA@EHHEHHGDEDEGG

Barcode sequences have been moved to the description of the ORF reads in the format `BC:Z:{barcode_sequence}`. 

This allows us to easily extract the barcode information from the aligned BAM file using tools like pysam.

## Alignment statistics

Get alignment statistics using samtools flagstat:

In [8]:
%%bash

samtools flagstat /media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_aligned.bam

10100 + 0 in total (QC-passed reads + QC-failed reads)
10000 + 0 primary
0 + 0 secondary
100 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
9686 + 0 mapped (95.90% : N/A)
9586 + 0 primary mapped (95.86% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


Very good alignment rates (95.86%), which means most ORF sequences can be confidently mapped to the cDNA sequences.

## Sorting alignments by barcode

Use `samtools sort` to sort the aligned BAM file by the barcode information in the description. This will allow us to group reads by their corresponding barcodes for downstream analysis:

In [13]:
%%bash

OUTDIR="/media/niek/4TB_SSD2/analyses/GPS_ONT"

samtools sort -@ 4 -o $OUTDIR/orfs_sorted_by_barcode.bam $OUTDIR/orfs_aligned.bam

samtools view $OUTDIR/orfs_sorted_by_barcode.bam | head -n 25

[bam_sort_core] merging from 0 files and 4 in-memory blocks...


e0af7a24-92af-4277-bb41-40bc2e2ab751	0	ENST00000379410.8|ENSG00000187583.12|OTTHUMG00000040756.5|OTTHUMT00000097940.2|PLEKHN1-203|PLEKHN1|3176|UTR5:1-50|CDS:51-1886|UTR3:1887-3176|	51	16	28M2D102M1D168M3D3M1D645M1I87M1D30M1I85M1I150M1D148M2D5M1I5M1D6M1D7M2I1M1I80M1D2M1D262M7S	*	0	0	ATGGGGAACAGCCACTGTGTGCCTCAGGCCCAGGAGGCTCCGGGCCTCCTTCTCCAGAAAGCCCTCGCTGAAGGGAAACAGAGAGGACAGCGCGCGGATGTCGGCCGGCCTGCCGGGCCCCGAGGCTGCTGAAGCGGGGACGCCGCCGCCAACAAGCTCTTCCACTACATCCCGGGCACGGACATCCTGGACCTGGAGAACCAGCGAGAAAACCTGGAGCAGCCATTCCTGAGTGTGTTCAAGAAGGGGCGGCGGAGGGTGCCTGTGAGGAACCTGGGAAAAGTTGTGCATTACGCCATTCGCTTCGGTTCCAGCACAGCCAGGATGTCAGCGACTGCTACCTGGAGCTATTCCCCGCCCACCTGTACTTCCAGGCCCACGGCTCGGAAGGACTCACATTTCAGGGGCTGTTACCGCTGACGGAGCTGAGTGTCTGCCCGCTCGAGGGGTCCCGAGAGCACGCCTTCCAGATCACAGGCCCACTGCCCGCACCCCTCCTGGTGCTCTGCCCCAGCCGGGCCGAGCTGGACCGCTGGCTTTACCACCTGGAGAAGCAGACGGCCCTCCTCGGGGGGCCGCGGCGCTGCCACTCGGCACCCCCACAGCGCCGTCTAACCCGGCTGCGGACGGCGTCAGGGCACGAACCCGGCGGCAGTGCTGTCTGTGCCTCGAGGGTCAAGCTGCAGCACCTGCCCGCACAGGAGCAGTGGGACCGGC